# Verifikasi Data Mart Bus Antarkota

Notebook ini memuat CSV hasil ETL ke SQLite in-memory. Database hanya digunakan selama notebook berjalan dan tidak menghasilkan file `.db`.

Sebelum menjalankan notebook, pastikan `pandas` dan Jupyter sudah tersedia pada environment Python yang dipilih.

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd

## Memuat CSV sebagai tabel

In [ ]:
candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PROJECT_ROOT = next(
    root for root in candidate_roots
    if (root / "data-mart" / "output").exists()
)
MART_DIR = PROJECT_ROOT / "data-mart" / "output"
connection = sqlite3.connect(":memory:")

csv_files = [
    *sorted((MART_DIR / "dimensions").glob("*.csv")),
    *sorted((MART_DIR / "facts").glob("*.csv")),
    *sorted((MART_DIR / "rejected").glob("*.csv")),
]

loaded_tables = []
for csv_file in csv_files:
    table_name = csv_file.stem
    dataframe = pd.read_csv(csv_file)
    dataframe.to_sql(table_name, connection, index=False, if_exists="replace")
    loaded_tables.append({"table_name": table_name, "row_count": len(dataframe)})

pd.DataFrame(loaded_tables)

In [ ]:
def run_query(sql):
    return pd.read_sql_query(sql, connection)

## Cara singkat membaca query

- `SELECT` menentukan kolom hasil.
- `FROM` menentukan tabel utama.
- `JOIN` mengambil keterangan dari tabel lain.
- `WHERE` menyaring baris.
- `GROUP BY` membentuk kelompok perhitungan.
- `ORDER BY` mengurutkan hasil.

## BQ 1 — Terminal transit dan rute dengan missed connection rate tertinggi

Rute masuk dan rute lanjutan dipasangkan menggunakan `itinerary_id` dan urutan leg.

In [ ]:
query_1 = """
SELECT
    t.terminal_name AS transit_terminal,
    route_in.route_name AS incoming_route,
    route_out.route_name AS onward_route,
    COUNT(*) AS total_connection_legs,
    SUM(f.missed_connection_count) AS total_missed_connections,
    ROUND(100.0 * SUM(f.missed_connection_count) / COUNT(*), 2)
        AS missed_connection_rate_pct
FROM fct_passenger_itinerary_leg AS f
JOIN fct_passenger_itinerary_leg AS previous_leg
    ON f.itinerary_id = previous_leg.itinerary_id
   AND f.leg_sequence = previous_leg.leg_sequence + 1
JOIN dim_terminal AS t
    ON f.origin_terminal_sk = t.terminal_sk
JOIN dim_route AS route_in ON previous_leg.route_sk = route_in.route_sk
JOIN dim_route AS route_out ON f.route_sk = route_out.route_sk
GROUP BY t.terminal_name, route_in.route_name, route_out.route_name
HAVING COUNT(*) >= 10
ORDER BY missed_connection_rate_pct DESC, total_connection_legs DESC
LIMIT 10
"""
result_1 = run_query(query_1)
result_1

## BQ 2 — Rute dan armada dengan arrival delay serta SLA breach rate tertinggi

Delay diringkas lebih dahulu per `trip_id` agar satu trip tidak dihitung berulang untuk setiap penumpang. Batas SLA yang digunakan adalah 60 menit.

In [ ]:
query_2 = """
WITH trip_performance AS (
    SELECT DISTINCT
        trip_id, route_sk, bus_fleet_sk, arrival_delay_minutes
    FROM fct_passenger_itinerary_leg
)
SELECT
    r.route_name,
    b.bus_fleet_id,
    b.bus_class,
    COUNT(*) AS total_trips,
    ROUND(AVG(tp.arrival_delay_minutes), 2) AS avg_arrival_delay_minutes,
    ROUND(100.0 * SUM(CASE WHEN tp.arrival_delay_minutes > 60 THEN 1 ELSE 0 END)
        / COUNT(*), 2) AS sla_breach_rate_pct
FROM trip_performance AS tp
JOIN dim_route AS r ON tp.route_sk = r.route_sk
JOIN dim_bus_fleet AS b ON tp.bus_fleet_sk = b.bus_fleet_sk
GROUP BY r.route_name, b.bus_fleet_id, b.bus_class
HAVING COUNT(*) >= 3
ORDER BY sla_breach_rate_pct DESC, avg_arrival_delay_minutes DESC
LIMIT 10
"""
result_2 = run_query(query_2)
result_2

## BQ 3 — Revenue leakage per rute selama peak season

Perhitungan hanya memakai klaim berstatus `Approved`. Baris yang berada di karantina tidak termasuk hasil ini.

In [ ]:
query_3 = """
SELECT
    r.route_name,
    SUM(f.refund_amount) AS total_refund,
    SUM(f.rebooking_cost) AS total_rebooking_cost,
    SUM(f.compensation_amount) AS total_compensation,
    SUM(f.refund_amount + f.rebooking_cost + f.compensation_amount)
        AS total_revenue_leakage
FROM fct_passenger_itinerary_leg AS f
JOIN dim_route AS r ON f.route_sk = r.route_sk
JOIN dim_date AS d ON f.scheduled_departure_date_sk = d.date_sk
WHERE d.is_peak_season = 1
  AND f.claim_status = 'Approved'
GROUP BY r.route_name
ORDER BY total_revenue_leakage DESC
LIMIT 10
"""
result_3 = run_query(query_3)
result_3

## BQ 4 — Frekuensi perjalanan pendapatan dan poin per loyalty tier

In [ ]:
query_4 = """
SELECT
    p.loyalty_tier,
    COUNT(DISTINCT f.itinerary_id) AS total_bookings,
    COUNT(DISTINCT p.passenger_id) AS total_passengers,
    ROUND(1.0 * COUNT(DISTINCT f.itinerary_id)
        / COUNT(DISTINCT p.passenger_id), 2) AS avg_booking_frequency,
    ROUND(AVG(f.allocated_base_fare - f.discount_amount), 2)
        AS avg_gross_ticket_revenue,
    SUM(f.loyalty_points_earned) AS total_points_earned,
    SUM(f.loyalty_points_redeemed) AS total_points_redeemed
FROM fct_ticket_sales AS f
JOIN dim_passenger AS p ON f.passenger_sk = p.passenger_sk
GROUP BY p.loyalty_tier
ORDER BY avg_booking_frequency DESC
"""
result_4 = run_query(query_4)
result_4

## BQ 5 — Missed connection dan repeat booking rate

Agregasi menggunakan `passenger_id` agar seluruh versi SCD milik penumpang yang sama tetap dihitung bersama. Bagian `WITH` hanya membuat dua ringkasan sementara dan tidak membuat tabel baru.

In [ ]:
query_5 = """
WITH disruption_by_passenger AS (
    SELECT
        p.passenger_id,
        MAX(f.missed_connection_count) AS ever_missed_connection
    FROM fct_passenger_itinerary_leg AS f
    JOIN dim_passenger AS p ON f.passenger_sk = p.passenger_sk
    GROUP BY p.passenger_id
),
booking_by_passenger AS (
    SELECT
        p.passenger_id,
        COUNT(DISTINCT f.booking_id) AS booking_count
    FROM fct_ticket_sales AS f
    JOIN dim_passenger AS p ON f.passenger_sk = p.passenger_sk
    GROUP BY p.passenger_id
)
SELECT
    CASE WHEN d.ever_missed_connection = 1
        THEN 'Pernah missed connection'
        ELSE 'Tidak pernah missed connection'
    END AS disruption_group,
    COUNT(*) AS passenger_count,
    ROUND(AVG(b.booking_count), 2) AS avg_booking_count,
    ROUND(100.0 * AVG(CASE WHEN b.booking_count > 1 THEN 1.0 ELSE 0.0 END), 2)
        AS repeat_booking_rate_pct
FROM disruption_by_passenger AS d
JOIN booking_by_passenger AS b ON d.passenger_id = b.passenger_id
GROUP BY d.ever_missed_connection
ORDER BY d.ever_missed_connection DESC
"""
result_5 = run_query(query_5)
result_5

## Data quality checks

In [ ]:
quality_query = """
SELECT 'duplicate ticket_sales_sk' AS check_name,
       COUNT(*) - COUNT(DISTINCT ticket_sales_sk) AS issue_count
FROM fct_ticket_sales
UNION ALL
SELECT 'duplicate itinerary_leg_sk',
       COUNT(*) - COUNT(DISTINCT itinerary_leg_sk)
FROM fct_passenger_itinerary_leg
UNION ALL
SELECT 'duplicate loyalty_transaction_sk',
       COUNT(*) - COUNT(DISTINCT loyalty_transaction_sk)
FROM fct_loyalty_transaction
UNION ALL
SELECT 'orphan foreign key on ticket sales', COUNT(*)
FROM fct_ticket_sales AS f
LEFT JOIN dim_passenger AS p ON f.passenger_sk = p.passenger_sk
LEFT JOIN dim_route AS r ON f.route_sk = r.route_sk
LEFT JOIN dim_sales_context AS s ON f.sales_context_sk = s.sales_context_sk
LEFT JOIN dim_promotion AS m ON f.promotion_sk = m.promotion_sk
WHERE p.passenger_sk IS NULL OR r.route_sk IS NULL
   OR s.sales_context_sk IS NULL OR m.promotion_sk IS NULL
UNION ALL
SELECT 'orphan foreign key on itinerary leg', COUNT(*)
FROM fct_passenger_itinerary_leg AS f
LEFT JOIN dim_passenger AS p ON f.passenger_sk = p.passenger_sk
LEFT JOIN dim_route AS r ON f.route_sk = r.route_sk
LEFT JOIN dim_bus_fleet AS b ON f.bus_fleet_sk = b.bus_fleet_sk
LEFT JOIN dim_terminal AS o ON f.origin_terminal_sk = o.terminal_sk
LEFT JOIN dim_terminal AS d ON f.destination_terminal_sk = d.terminal_sk
WHERE p.passenger_sk IS NULL OR r.route_sk IS NULL OR b.bus_fleet_sk IS NULL
   OR o.terminal_sk IS NULL OR d.terminal_sk IS NULL
UNION ALL
SELECT 'orphan foreign key on loyalty transaction', COUNT(*)
FROM fct_loyalty_transaction AS f
LEFT JOIN dim_passenger AS p ON f.passenger_sk = p.passenger_sk
LEFT JOIN dim_loyalty_transaction_type AS t
    ON f.loyalty_transaction_type_sk = t.loyalty_transaction_type_sk
WHERE p.passenger_sk IS NULL OR t.loyalty_transaction_type_sk IS NULL
UNION ALL
SELECT 'negative loyalty balance', COUNT(*)
FROM fct_loyalty_transaction
WHERE points_balance_after < 0
UNION ALL
SELECT 'unexpected null on required fact columns',
       (SELECT COUNT(*) FROM fct_ticket_sales
        WHERE ticket_number IS NULL OR passenger_sk IS NULL OR route_sk IS NULL)
       +
       (SELECT COUNT(*) FROM fct_passenger_itinerary_leg
        WHERE ticket_number IS NULL OR passenger_sk IS NULL OR trip_id IS NULL)
UNION ALL
SELECT 'trip above historical capacity', COUNT(*)
FROM (
    SELECT f.trip_id
    FROM fct_passenger_itinerary_leg AS f
    JOIN dim_bus_fleet AS b ON f.bus_fleet_sk = b.bus_fleet_sk
    GROUP BY f.trip_id, b.seat_capacity
    HAVING COUNT(*) > b.seat_capacity
) AS over_capacity_trip
UNION ALL
SELECT 'multiple current passenger versions', COUNT(*)
FROM (
    SELECT passenger_id
    FROM dim_passenger
    GROUP BY passenger_id
    HAVING SUM(CASE WHEN is_current = 1 THEN 1 ELSE 0 END) <> 1
) AS invalid_current_version
UNION ALL
SELECT 'itinerary reconciliation difference',
       25000 - ((SELECT COUNT(*) FROM fct_passenger_itinerary_leg)
              + (SELECT COUNT(*) FROM rejected_itinerary_legs))
"""
quality_results = run_query(quality_query)
quality_results

In [ ]:
# Tutup koneksi setelah seluruh pemeriksaan selesai.
connection.close()